# PXD024871 — container extraction

Downloads the 52 class-I identification containers, verifies each against the
publisher's checksum, extracts its peptide table, deletes the container, and
pushes the extracts back. **45 GiB of transfer; nothing else.**

## This job does not use a GPU

Not "does not need one" — *cannot use one*. The work is a network transfer, a
SHA-1 (sequential by construction), one SQLite `GROUP BY`, and a CSV write. None
of it is parallel arithmetic. Measured on a real container: 315 s total, of
which ~305 s was the download and about 1 s the extraction. A GPU would sit
idle and still cost you compute units.

**Runtime → Change runtime type → CPU.** The next cell will tell you if you got
this wrong.

## You may not want Colab at all

These scripts are standard-library Python plus `curl` and `git`. They run on
your own laptop with no session limit, no disconnects, and no compute units:

```bash
git clone -b claude/mapping-research-project-prompt-glsess https://github.com/hafijbd/PXD024871_mapping.git
cd PXD024871_mapping && python3 scripts/batch_extract.py
```

Colab's advantage here is bandwidth, and that is **unverified** — the local
session measured 1.6 MiB/s and Colab is probably faster, but nobody has
measured it. Cell 4 reports the rate so you can decide after one container.

## Token

Fine-grained PAT at `github.com/settings/personal-access-tokens/new`: this
repository only, **Contents: Read and write**, short expiry. Add to Colab
Secrets (key icon) as **`GH_TOKEN`** with notebook access on. Revoke it when
the run finishes.

## No dependencies are installed

Extraction needs only the standard library. The previous version installed
numpy and scipy, which this job never imports and which conflict with Colab's
preinstalled numba. They are needed for the analysis stages, not this one.

In [ ]:
#@title 1 — Check the runtime
import os, shutil, subprocess

gpu = shutil.which("nvidia-smi") and subprocess.run(
    ["nvidia-smi","--query-gpu=name","--format=csv,noheader"],
    capture_output=True, text=True).stdout.strip()

print(f"Colab    : {os.environ.get('COLAB_RELEASE_TAG', 'not Colab')}")
print(f"CPUs     : {os.cpu_count()}")
print(f"Disk free: {shutil.disk_usage('/content').free / 2**30:.1f} GiB  (need ~12 GiB peak)")
print(f"Python   : {subprocess.run(['python3','--version'],capture_output=True,text=True).stdout.strip()}")
print()
if gpu:
    print("=" * 68)
    print(f"  WRONG RUNTIME: {gpu} attached.")
    print("  This job cannot use a GPU. You are paying compute units for an")
    print("  accelerator that will stay idle for the entire run.")
    print("  Runtime > Change runtime type > CPU, then re-run from here.")
    print("=" * 68)
else:
    print("CPU runtime — correct for this job.")

In [ ]:
#@title 2 — Clone or update the repository
from google.colab import userdata
import os, pathlib, subprocess

REPO   = "hafijbd/PXD024871_mapping"
BRANCH = "claude/mapping-research-project-prompt-glsess"
WORK   = pathlib.Path("/content/PXD024871_mapping")

os.environ["GH_TOKEN"] = userdata.get("GH_TOKEN")
# Supplied per command so the token never lands in .git/config. It is needed on
# every git call against this private repo, not just the clone.
HELPER = '!f() { echo username=x-access-token; echo "password=$GH_TOKEN"; }; f'
GIT = ["git", "-c", f"credential.helper={HELPER}"]

if WORK.exists():
    subprocess.run(GIT + ["-C", str(WORK), "pull", "--rebase", "origin", BRANCH], check=True)
else:
    subprocess.run(GIT + ["clone", "--branch", BRANCH,
                          f"https://github.com/{REPO}.git", str(WORK)], check=True)
os.chdir(WORK)
subprocess.run(["git","config","user.name","Claude Code (Colab runner)"], check=True)
subprocess.run(["git","config","user.email","noreply@anthropic.com"], check=True)
print(subprocess.run(["git","log","--oneline","-1"],capture_output=True,text=True).stdout)
print(subprocess.run(["python3","-I","scripts/batch_extract.py","--dry-run"],
                     capture_output=True,text=True).stdout.split("\n")[1])

In [ ]:
#@title 3 — Record this machine as its own env_id
ENV_ID = "env-002"  #@param {type:"string"}

out = subprocess.run(["python3","-I","scripts/capture_environment.py",ENV_ID],
                     capture_output=True,text=True).stdout
if f"## {ENV_ID} " in pathlib.Path("ENVIRONMENT.md").read_text():
    print(f"{ENV_ID} already recorded — skipping (re-running this cell is safe)")
else:
    with open("ENVIRONMENT.md","a") as fh: fh.write(out)
    print(out)

In [ ]:
#@title 4 — Extract, with live progress. Re-runnable: completed units are skipped.
CHUNK    = 0  #@param {type:"integer"}
PARALLEL = 4  #@param {type:"integer"}
REFRESH  = 5  #@param {type:"integer"}
# CHUNK 0 = everything remaining. PARALLEL = concurrent downloads; EBI
# rate-limits per connection so this is close to linear, and the script
# reduces it itself if free disk cannot hold that many at once.

import glob, json, os, subprocess, time
from pathlib import Path
from IPython.display import clear_output

LOG = Path("batch_run.log")
args = (["python3", "-u", "-I", "scripts/batch_extract.py", "--parallel", str(PARALLEL)]
        + (["--limit", str(CHUNK)] if CHUNK else []))

sizes = {}
for pg in sorted(Path("data/raw/S1").glob("files_v3_page*.json")):
    for r in json.loads(pg.read_text()):
        if r["fileName"].endswith("_class_I.msf"):
            sizes[r["fileName"][:-len("_class_I.msf")]] = r["fileSizeBytes"]
TOTAL = sum(sizes.values())

def done_units():
    return {Path(f).name.split(".")[0] for f in glob.glob("data/derived/peptides/*.csv*")}

start_done = done_units()
start_bytes = sum(sizes[u] for u in start_done if u in sizes)
t0 = time.time()

with LOG.open("w") as fh:
    proc = subprocess.Popen(args, stdout=fh, stderr=subprocess.STDOUT)
    try:
        while True:
            alive = proc.poll() is None
            now = done_units()
            got = sum(sizes[u] for u in now if u in sizes)
            inflight = sorted(Path("data/raw/S3").glob("*.msf"))
            partial = sum(f.stat().st_size for f in inflight)
            moved = (got - start_bytes) + partial
            el = time.time() - t0
            rate = moved / el if el > 0 else 0
            left = TOTAL - got
            pct = 100 * got / TOTAL

            clear_output(wait=True)
            bar = "#" * int(pct / 2.5) + "." * (40 - int(pct / 2.5))
            print(f"[{bar}] {pct:5.1f}%")
            print(f"units    {len(now)}/52      remaining {left/2**30:6.2f} GiB")
            print(f"rate     {rate/2**20:5.2f} MiB/s   elapsed {el/60:5.1f} min", end="")
            print(f"   ETA {left/rate/60:6.1f} min" if rate > 1000 else "   ETA --")
            if inflight:
                print(f"in flight ({len(inflight)}):")
                for f in inflight:
                    want = sizes.get(f.name[:-len('_class_I.msf')], 0)
                    have = f.stat().st_size
                    print(f"  {f.name:<24} {have/2**20:>6,.0f}/{want/2**20:>6,.0f} MiB"
                          f"  {100*have/want if want else 0:3.0f}%")
            tail = LOG.read_text().splitlines()[-6:] if LOG.exists() else []
            if tail:
                print("-" * 72)
                for ln in tail:
                    print(" ", ln[:110])
            if not alive:
                print("-" * 72)
                print(f"FINISHED in {el/60:.1f} min — {len(now)}/52 units")
                break
            time.sleep(REFRESH)
    except KeyboardInterrupt:
        proc.terminate()
        print("\nstopped — completed units are kept; re-run to resume")

In [ ]:
#@title 5 — Commit and push. Run after every chunk, and after stopping cell 4.
import glob, subprocess
before = len(glob.glob("data/derived/peptides/*.csv.gz"))
subprocess.run(["bash", "scripts/commit_extracts.sh"], check=False)
r = subprocess.run(GIT + ["push", "origin", BRANCH], capture_output=True, text=True)
after = len(glob.glob("data/derived/peptides/*.csv.gz"))
print(f"\n{after}/52 units extracted  (+{after-before} this push)")
if r.returncode != 0:
    print("PUSH FAILED — extracts are committed locally but not safe yet:")
    print(r.stderr.strip()[:400])
else:
    print("pushed — safe if this runtime disconnects")

## Running it to completion

Repeat **cell 4 → cell 5** until cell 5 reports 52/52. Set `CHUNK = 0` once you
know the rate and are confident the session will hold.

If the session drops: re-run cells 2 → 5. `batch_extract.py` skips finished
units and reconciles any half-transferred container before anything reads it, so
a dropped session costs only the container that was in flight.

## Watch for

- **`CHECKSUM_MISMATCH`** — a truncated transfer. The file is rejected, not
  extracted. Re-run to retry it. One occurred during local testing, so expect
  the occasional one.
- **`ABORT: container stems do not match`** — the repository is missing the S1
  manifest. Pull and re-run cell 2.

## What this notebook will not do

It does not fit models, choose thresholds, or construct negatives. Those are
preregistration-gated (`DECISION_LOG.md` D002, D004) and running them from a
notebook cell would put untraceable analysis in front of the gates built to
prevent exactly that.

Training is a separate job on a **T4** — the one stage that is genuinely matrix
arithmetic — recorded under its own `env_id`, after re-running
`scripts/benchmark_compute.py` on that hardware. It does not exist yet: G4 has
not been passed.